Changes in the HE runs processing to:
* Delete COMPLETELY non desired hits (out of Z range) before doing corrections --> we are not taking into account their energy at all
* The hits INSIDE can be redistributed using the Q per slice. Am I doing it? Not right now, as 
* Use final 3D map function given by Gonzalo
* Delete wf selection part as it is a bit messy --> CHECK

In [177]:
import argparse
import glob
import numpy as np
import pandas as pd
import tables as tb
from   typing      import Callable
from   typing      import Optional
from   typing      import List
from   pandas      import DataFrame
from   pandas      import Series

from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN

from scipy.interpolate import griddata

from invisible_cities.reco.corrections import read_maps, apply_all_correction
from invisible_cities.types.symbols    import NormStrategy
from invisible_cities.types.ic_types   import NN

def get_args():
    parser = argparse.ArgumentParser(
        description="Process a NEXT100 run and build summaries")
    parser.add_argument(
        "-r", "-run", "--run",
        dest="run_number",
        type=int,
        required=True,
        help="Run number to analyse, e.g. 15107")
    parser.add_argument(
        "-m", "--map",
        dest="map_name",
        type=str,
        default='map_MC_4bar_15063.h5',
        help="Name of the correction map")
    parser.add_argument(
        "-d", "--dim",
        dest="dimension",
        type=int,
        default=3,
        help="Number of dimensions for clustering algorithm")
    parser.add_argument(
        "-s", "--save",
        dest="save_name",
        type=str,
        default="run_summary.h5",
        help="Name of the file to save")
    parser.add_argument(
        "-l", "--ldc",
        dest="ldc_name",
        type=str,
        default="*",
        help="Name of the ldc to run over")
    parser.add_argument(
        "-n", "--nhit",
        dest="drop_nhits",
        type=int,
        default=3,
        help="Number of hits to drop a cluster")
    parser.add_argument(
        "-c", "--corr",
        dest="corr_type",
        type=str,
        default="2D",
        help="Type of map, 2D or 3D")
    parser.add_argument(
        "-q", "--q_cut",
        dest="q_thr",
        type=float,
        default=0,
        help="Q cut to apply"
    )
    return parser.parse_args()

# args        = get_args()
run_number  = 15604 #args.run_number 
map_name    = 'combined_15546_15557.map3d' #args.map_name
save_name   = '' #args.save_name
ldc_name    = 'ldc1' #args.ldc_name
drop_nhits  = 3 #args.drop_nhits
corr_type   = "3D" #args.corr_type
q_thr       = 7 #args.q_thr

#just add ldc tag to saved file if ldc is not all
if ldc_name != '*':
    save_name = ldc_name + save_name

drop_cluster_dim = 3 #args.dimension


source_path = '/mnt/netapp1/Store_next_data/NEXT100/data/{run_n}/hdf5/prod_Qth5PE/*/*/sophronia/trigger2/'.format(run_n = run_number)
data_path = source_path + '{}/*'.format(ldc_name)

store_path  = '/mnt/lustre/scratch/nlsas/home/usc/ie/mpm/NEXT100/data/HE_ana_runs/' 
save_path_summary = store_path + '/{run_n}/'.format(run_n = run_number) + save_name
map_path = store_path + map_name

def lineal(x, a, b):
    return a * x + b

def in_range(x, min_, max_):
    return (x > min_) & (x < max_)

def get_fname_info(path):
    file = path.split('/')[-1]
    split_name = file.split('_')
    run_n, file_n, ldc_n = split_name[1], split_name[2], split_name[3]
    return run_n, file_n, ldc_n

def redistribute_energy(group: pd.DataFrame, var: str) -> pd.DataFrame:
    """
    Funtion that redistributes energy per slice given the Q of the SiPMs
    """
    tot_E = group.E.sum()
    # mask = group['drop'].values
    mask = group[var].values < 0
    drp = group[mask]
    srv = group[~mask]

    if drp.empty:
        # no hits to distribute
        return srv

    if srv.empty:
        # hits turned NN
        drp = drp.copy()
        drp[['X', 'Y', 'Q']] = NN  # or NN if defined
        return drp

    # redistribute
    srv = srv.copy()
    srv['E'] = (srv.Q / srv.Q.sum()) * tot_E
    return srv

def merge_NN_hits(hits: pd.DataFrame, same_peak: bool = True) -> pd.DataFrame:
    # quickly split NN vs normal
    sel = hits.Q.eq(NN)
    if not sel.any():
        return hits

    normal = hits[~sel].copy()
    nn     = hits[sel]

    if normal.empty:
        # nothing to receive: drop all NN
        return normal

    # For each NN, find candidate receivers and closest distance
    # Build a mapping of receiver index -> energy/energy_correction
    corr = pd.DataFrame(0.0, index=normal.index, columns=["E", "Ec"])

    # precompute distances matrix only once if same_peak=False
    if same_peak:
        # group normal hits by npeak to avoid repeated filtering
        normal_groups = {p: g for p, g in normal.groupby("npeak")}
    else:
        z_normal = normal.Z.values
        idx_normal = normal.index.values

    for _, row in nn.iterrows():   # still a loop over NN hits, but usually few
        if same_peak:
            cand = normal_groups.get(row.npeak)
            if cand is None or cand.empty:
                continue
            dz = (cand.Z - row.Z).abs()
            closest = cand.loc[np.isclose(dz, dz.min())]
        else:
            dz = np.abs(z_normal - row.Z)
            m  = np.isclose(dz, dz.min())
            closest = normal.loc[idx_normal[m]]

        wE  = closest.E / closest.E.sum()
        wEc = closest.Ec / closest.Ec.sum()
        corr.loc[closest.index, "E"]  += row.E  * wE
        corr.loc[closest.index, "Ec"] += row.Ec * wEc

    normal[["E","Ec"]] += corr
    return normal

def make_Q_cut(df: pd.DataFrame, q_thr: float = 7):
    df.loc[df.Q < q_thr, 'pass'] = -1
    df.loc[df.Q >= q_thr, 'pass'] = 0

    df = (df.groupby(['Z'], group_keys=False)
                        .apply(redistribute_energy, "pass")
                        .reset_index(drop=True))
    df = merge_NN_hits(df)
    return df

def drop_isolated_clusters(distance: List[float] = [16., 16., 4.], nhit: int = 3) -> Callable:
    '''
    If len(distance) == 2, it will perform on X, Y
    If len(distance) == 3, it will perform on X, Y, Z
    '''
    ndim = len(distance)
    dist = np.sqrt(ndim)

    def drop(df: pd.DataFrame) -> pd.DataFrame:
        if len(df) == 0:
            return df
        
        # # apply threshold to separate the hits before clusterizing
        # df, dropQ = make_Q_cut(df, q_thr)
        # # dropQ['drop'] = True
        # dropQ['cluster_id'] = -1

        coords = []

        coords.append(df.X.values / distance[0])
        coords.append(df.Y.values / distance[1])

        if ndim == 3:
            coords.append(df.Z.values / distance[2])

        coords = np.column_stack(coords)

        try:
            nbrs = NearestNeighbors(radius=dist, algorithm='ball_tree').fit(coords)
            neighbors = nbrs.radius_neighbors(coords, return_distance=False)
            mask = np.array([len(neigh) > nhit for neigh in neighbors]) # ADAPT THIS TO CLUSTERIZE EVENTS!!
            # db = DBSCAN(eps=dist, min_samples=nhit, algorithm="ball_tree")
            # labels = db.fit_predict(coords)
        except Exception as e:
            print(f"Error in NearestNeighbors: {{e}}")
            return df.iloc[:0]  # fallback: return empty

        # # add cluster id labels to all data
        # df['cluster_id'] = labels 
        # # separate to get the in range dropped
        # mask = labels != -1
        pass_df = df.loc[mask].copy()
        drop_df = df.loc[~mask].copy()
        # mask them to redistribute their energy after 
        pass_df['drop'] = 0
        drop_df['drop'] = -1

        # recover the hits that are inside the Z range, because we want their energy to be taken into account
        # the energy of the out of range will be simply discarded
        if not pass_df.empty:
            zmin, zmax = pass_df.Z.min(), pass_df.Z.max()
            inside_mask = (drop_df.Z >= zmin) & (drop_df.Z <= zmax)
            drop_inrange = drop_df.loc[inside_mask]
            if not drop_inrange.empty:
                pass_df = pd.concat([pass_df, drop_inrange], axis=0)

        
        # # now append the hits that did not pass the Q cut to redistribute their energy
        # pass_df = pd.concat([pass_df, dropQ], axis=0)
        # at this point I have the df with the energy I should use (ener outsize Z range is deleted)
        # now we redistribute the energy of the dropped in range hits per slice using the charge
        pass_df = (pass_df.groupby(['Z'], group_keys=False)
                        .apply(redistribute_energy, "drop") #CHANGE TO CLUSTER ID
                        .reset_index(drop=True))
        # and finally redistribute the energy of the hits that were alone in a slice
        pass_df = merge_NN_hits(pass_df)
        return pass_df

    return drop

def get_corr(filename):
    krmap = pd.read_hdf(filename, "/krmap")
    meta  = pd.read_hdf(filename, "/mapmeta")
    dtxy_map   = krmap.loc[:, list("zxy")].values
    factor_map = krmap.factor.values
    # t is not used but for the other option
    def corr(x, y, dt, t, method="nearest"):
        dtxy_data   = np.stack([dt, x, y], axis=1)
        factor_data = griddata(dtxy_map, factor_map, dtxy_data, method=method)
        return factor_data
    return corr

def hits_summary(group, fname, coords = ['X', 'Y', 'Z'], ener = 'Ec'):
    def dcoord(group, coords, i):
        c = coords[i]
        return group[c].max() - group[c].min()
    def barycenter(group, coords, i):
        return (group[coords[i]] * group['Q']).sum() / group['Q'].sum() #changed to Q because makes more sense
    
    run_n, file_n, ldc_n = get_fname_info(fname)
    
    return pd.Series({
        'run_n': str(run_n),
        'file_n': str(file_n),
        'ldc_n': str(ldc_n),
        'time': group['time'].unique()[0],
        'dX': dcoord(group, coords, 0),
        'dY': dcoord(group, coords, 1),
        'dZ': dcoord(group, coords, 2),
        'Xmin': group[coords[0]].min(),
        'Ymin': group[coords[1]].min(),
        'Zmin': group[coords[2]].min(), 
        'Rmax': np.sqrt(group['X']**2 + group['Y']**2).max(),
        'Xmax': group[coords[0]].max(),
        'Ymax': group[coords[1]].max(),
        'Zmax': group[coords[2]].max(),
        'X_bary': barycenter(group, coords, 0),
        'Y_bary': barycenter(group, coords, 1),
        'Z_bary': barycenter(group, coords, 2),
        'total_E': group['E'].sum(),
        'total_energy': group[ener].sum(),
        'num_hits': int(len(group)),
    })


def create_hits_summary(reco, corr_fun, fname):
    # Correct energy
    factor = corr_fun(reco.X, reco.Y, reco.Z, reco.time)
    #correct energy in the borders (for NaN hits)
    factor_border = corr_fun([479],[0],[0],[1])
    reco['Ec'] = reco.E * factor
    reco["Ec_border"] = reco.E * factor_border
    # add the latter energy
    reco["Ec"] = reco["Ec"].fillna(reco["Ec_border"])
    # do summary
    reco_summary = reco.groupby(['event', 'npeak']).apply(lambda group: hits_summary(group, fname)).reset_index() #groupby cluster id too

    return reco_summary

files = sorted(glob.glob(data_path + '*'), key=lambda x: (x.split('/')[-2], int(x.split('/')[-1].split('_')[2])))

# Energy correction
if corr_type == "2D":
    maps = read_maps(map_path)

    get_coef  = apply_all_correction(maps
                                    , apply_temp = False
                                    , norm_strat = NormStrategy.kr)
if corr_type == "3D":
    get_coef = get_corr(map_path)

# Cluster dropping
if drop_cluster_dim == 2:
    dist = [16., 16.]
if drop_cluster_dim == 3:
    dist = [16., 16., 4.]

dropper = drop_isolated_clusters(distance = dist, nhit = drop_nhits)

# time_to_Z = get_df_to_z_converter(maps) if maps.t_evol is not None else identity

for i, f in enumerate(files):
    nev = []
    try:
        dst = pd.read_hdf(f, 'DST/Events')
        reco = pd.read_hdf(f, 'RECO/Events')
    except Exception as e:
        print(f"Skipping corrupted/invalid file: {f}")
        print(f"Error: {e}")
        continue

    if reco.empty:
        print(f"Skipping empty file: {f}")
        continue
    
    # apply Q cut
    reco = reco.groupby(['event', 'npeak'], group_keys=False).apply(make_Q_cut, q_thr)
    # drop hits and redistribute energy
    reco = reco.groupby(['event', 'npeak'], group_keys=False).apply(dropper)
    # correct energy and create summary
    reco_summary = create_hits_summary(reco, get_coef, f)
        
    # dst.to_hdf (save_path_summary, key = 'DST/Events', mode = 'a', append= True, complib="zlib", complevel=4)
    # reco_summary.to_hdf(save_path_summary, key = 'RECO/Events_summary', mode = 'a', append= True, complib="zlib", complevel=4)

    print(i)
    break

0


## Para el clustering algorithm cambié el propio neares neighbors por DBSCAN, con el añado una nueva label llamada "cluster_id". Luego la mascara la hago en base a que los hits -1 son los que elimino (cambio tambien esa mascara en redistribute energy), y finalmente cuando hago el summary tengo que agrupar también por "cluster_id". Esto me deja aun algo suelto, que es cómo aplicar los cortes (los aplico así directamente a cada cluster, de esta forma podría en un evento tener un cluster que pasa cierto corte y otro que no, o debería aplicar corte por evento completo como antes igualmente?)

In [29]:
reco_summary[reco_summary.event == 645].sum()

event                           2580
npeak                             85
cluster_id                         3
run_n           15604156041560415604
file_n              0000000000000000
ldc_n               ldc1ldc1ldc1ldc1
time                  7011015510.548
dX                            354.65
dY                             432.4
dZ                         84.830125
Xmin                          -61.95
Ymin                          -313.5
Zmin                      3343.08175
Rmax                      980.781943
Xmax                           292.7
Ymax                           118.9
Zmax                     3427.911875
X_bary                    101.807459
Y_bary                    -80.010903
Z_bary                   3391.437435
total_E                183741.290278
total_energy           190229.664703
num_hits                         666
dtype: object

In [27]:
reco_summary_old[reco_summary_old.event == 645].sum()

event                     1290
npeak                       43
run_n               1560415604
file_n                00000000
ldc_n                 ldc1ldc1
time            3505507755.274
dX                       401.3
dY                       585.9
dZ                  138.359625
Xmin                     -84.9
Ymin                    -140.7
Zmin                 1749.0015
Rmax                515.602962
Xmax                     316.4
Ymax                     445.2
Zmax               1887.361125
X_bary              174.393634
Y_bary              272.613223
Z_bary              1790.39748
total_E          183741.290278
total_energy     190230.671931
num_hits                   665
dtype: object

In [21]:
dst[dst.event == 645]

,event,time,s1_peak,s2_peak,nS1,nS2,S1w,S1h,S1e,S1t,...,Nsipm,DT,Z,Zrms,X,Y,R,Phi,Xrms,Yrms
101,645,1.752754e+09,0,0,3,2,350.0,22.128246,143.221390,648275.0,...,3534,760.210510,760.210510,28.320340,29.012382,27.671458,40.092741,0.761746,239.852854,237.006632
102,645,1.752754e+09,0,1,3,2,350.0,22.128246,143.221390,648275.0,...,2579,1025.208618,1025.208618,8.571356,8.541730,46.580623,47.357318,1.389436,235.637221,245.712914
103,645,1.752754e+09,1,0,3,2,325.0,7.556017,42.388363,864600.0,...,3534,543.885498,543.885498,28.320340,29.012382,27.671458,40.092741,0.761746,239.852854,237.006632
104,645,1.752754e+09,1,1,3,2,325.0,7.556017,42.388363,864600.0,...,2579,808.883667,808.883667,8.571356,8.541730,46.580623,47.357318,1.389436,235.637221,245.712914
105,645,1.752754e+09,2,0,3,2,350.0,11.572730,78.649216,1160425.0,...,3534,248.060516,248.060516,28.320340,29.012382,27.671458,40.092741,0.761746,239.852854,237.006632
106,645,1.752754e+09,2,1,3,2,350.0,11.572730,78.649216,1160425.0,...,2579,513.058655,513.058655,8.571356,8.541730,46.580623,47.357318,1.389436,235.637221,245.712914


In [7]:
reco_summary

,event,npeak,run_n,file_n,ldc_n,time,dX,dY,dZ,Xmin,...,Rmax,Xmax,Ymax,Zmax,X_bary,Y_bary,Z_bary,total_E,total_energy,num_hits
0,29,32,15604,0000,ldc1,1.752754e+09,46.65,45.65,11.276625,-312.575,...,318.776977,-265.925,-32.475,631.869125,-292.728372,-50.893879,626.597295,5676.685660,5915.281109,27
1,29,33,15604,0000,ldc1,1.752754e+09,92.30,107.85,30.951625,-373.775,...,384.675326,-281.475,106.475,702.719125,-331.425144,44.332359,686.496662,68410.111426,73869.801950,203
2,29,59,15604,0000,ldc1,1.752754e+09,93.30,92.30,19.367000,-96.875,...,405.299002,-3.575,-309.875,1986.961750,-44.116126,-358.781309,1978.357101,15517.390377,17494.489434,92
3,50,33,15604,0000,ldc1,1.752754e+09,61.20,62.20,10.907000,-265.925,...,300.677919,-204.725,-124.275,303.483625,-233.353989,-146.312320,297.672982,6611.564345,6873.637547,39
4,50,35,15604,0000,ldc1,1.752754e+09,107.85,123.40,23.103375,-343.675,...,371.219088,-235.825,-32.475,414.534875,-298.993922,-94.141208,402.157427,59934.818369,62076.510830,168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,645,21,15604,0000,ldc1,1.752754e+09,293.45,446.95,115.101500,-81.325,...,259.022202,212.125,199.275,848.015000,126.347752,84.677328,763.101108,137899.738360,142691.289310,479
71,645,22,15604,0000,ldc1,1.752754e+09,107.85,138.95,23.258125,-3.575,...,256.580760,104.275,245.925,1039.346125,47.957818,187.728926,1027.355503,45841.551917,47538.375394,187
72,687,19,15604,0000,ldc1,1.752754e+09,46.65,62.20,15.316375,273.325,...,433.234124,319.975,307.625,808.176000,289.393159,278.743730,799.540342,5927.478443,6866.413116,26
73,687,23,15604,0000,ldc1,1.752754e+09,170.05,215.70,179.363875,-467.075,...,483.454089,-297.025,-32.475,1251.399750,-380.197526,-145.662179,1140.907922,283239.846632,311545.954997,1212


In [6]:
reco_old

,event,time,npeak,Xpeak,Ypeak,nsipm,X,Y,Xrms,Yrms,Z,Q,E,Qc,Ec,track_id,Ep,drop,Ec_border
0,29,1.752754e+09,32,-10.893259,25.457517,1,-312.575,-48.025,0.0,0.0,620.592500,11.336324,1007.452496,-1.0,1058.499190,-1,-1.0,False,1479.051964
1,29,1.752754e+09,32,-10.893259,25.457517,1,-297.025,-48.025,0.0,0.0,620.592500,8.926717,793.312129,-1.0,825.987929,-1,-1.0,False,1164.670164
2,29,1.752754e+09,32,-10.893259,25.457517,1,-312.575,-62.575,0.0,0.0,624.352000,7.421574,118.056134,-1.0,124.472592,-1,-1.0,False,173.319494
3,29,1.752754e+09,32,-10.893259,25.457517,1,-297.025,-62.575,0.0,0.0,624.352000,7.558621,120.236167,-1.0,124.710574,-1,-1.0,False,176.520024
4,29,1.752754e+09,32,-10.893259,25.457517,1,-281.475,-78.125,0.0,0.0,624.352000,7.331844,116.628781,-1.0,120.544538,-1,-1.0,False,171.223981
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
925,722,1.752754e+09,24,137.563008,72.314694,1,381.175,184.225,0.0,0.0,253.675375,23.159225,344.142590,-1.0,410.945527,-1,-1.0,False,505.239479
926,722,1.752754e+09,24,137.563008,72.314694,1,396.725,168.675,0.0,0.0,253.675375,7.958416,118.260859,-1.0,139.215811,-1,-1.0,False,173.620053
927,722,1.752754e+09,24,137.563008,72.314694,1,381.175,198.775,0.0,0.0,253.675375,17.003962,252.676299,-1.0,307.183597,-1,-1.0,False,370.956822
928,722,1.752754e+09,24,137.563008,72.314694,1,366.625,184.225,0.0,0.0,257.784625,8.378812,277.776679,-1.0,321.145302,-1,-1.0,False,407.806963


In [2]:
reco_old = reco.copy()
reco_summary_old = reco_summary.copy()

event                     1290
npeak                       43
run_n               1560415604
file_n                00000000
ldc_n                 ldc1ldc1
time            3505507755.274
dX                       401.3
dY                       585.9
dZ                  138.359625
Xmin                     -84.9
Ymin                    -140.7
Zmin                 1749.0015
Rmax                515.602962
Xmax                     316.4
Ymax                     445.2
Zmax               1887.361125
X_bary              174.393634
Y_bary              272.613223
Z_bary              1790.39748
total_E          183741.290278
total_energy     190230.671931
num_hits                   665
dtype: object

In [16]:
reco

,event,time,npeak,Xpeak,Ypeak,nsipm,X,Y,Xrms,Yrms,Z,Q,E,Qc,Ec,track_id,Ep,drop,Ec_border
38,8,1.752887e+09,13,83.177390,71.828271,1,351.075,276.525,0.0,0.0,46.113750,8.308585,197.450264,-1.0,245.209152,-1,-1.0,False,289.878880
39,8,1.752887e+09,13,83.177390,71.828271,1,366.625,245.425,0.0,0.0,46.113750,7.239820,172.051469,-1.0,208.382766,-1,-1.0,False,252.590632
40,8,1.752887e+09,13,83.177390,71.828271,1,366.625,260.975,0.0,0.0,46.113750,13.626697,323.833094,-1.0,412.308381,-1,-1.0,False,475.422887
41,8,1.752887e+09,13,83.177390,71.828271,1,366.625,276.525,0.0,0.0,46.113750,10.682334,253.861472,-1.0,331.727456,-1,-1.0,False,372.696788
42,8,1.752887e+09,13,83.177390,71.828271,1,366.625,292.075,0.0,0.0,46.113750,11.596043,275.575400,-1.0,405.241254,-1,-1.0,False,404.575242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,666,1.752887e+09,25,104.868816,81.161743,1,212.125,152.625,0.0,0.0,636.629063,11.778493,117.258246,-1.0,120.834051,-1,-1.0,False,172.148107
1176,666,1.752887e+09,25,104.868816,81.161743,1,149.925,137.075,0.0,0.0,640.573438,7.898041,306.215167,-1.0,318.943546,-1,-1.0,False,449.557817
1177,666,1.752887e+09,25,104.868816,81.161743,1,181.025,137.075,0.0,0.0,640.573438,7.096191,275.126625,-1.0,281.641003,-1,-1.0,False,403.916391
1178,666,1.752887e+09,25,104.868816,81.161743,1,181.025,152.625,0.0,0.0,640.573438,10.918341,423.315277,-1.0,431.313420,-1,-1.0,False,621.473761


In [14]:
reco

,event,time,npeak,Xpeak,Ypeak,nsipm,X,Y,Xrms,Yrms,Z,Q,E,Qc,Ec,track_id,Ep,drop,Ec_border
0,8,1.752887e+09,13,83.177390,71.828271,1,335.525,260.975,0.0,0.0,46.113750,5.421461,70.251118,-1.0,81.876018,-1,-1.0,False,103.136430
1,8,1.752887e+09,13,83.177390,71.828271,1,351.075,260.975,0.0,0.0,46.113750,5.076050,65.775289,-1.0,79.906256,-1,-1.0,False,96.565417
2,8,1.752887e+09,13,83.177390,71.828271,1,351.075,276.525,0.0,0.0,46.113750,8.308585,107.662378,-1.0,133.703545,-1,-1.0,False,158.060307
3,8,1.752887e+09,13,83.177390,71.828271,1,366.625,245.425,0.0,0.0,46.113750,7.239820,93.813347,-1.0,113.623469,-1,-1.0,False,137.728395
4,8,1.752887e+09,13,83.177390,71.828271,1,366.625,260.975,0.0,0.0,46.113750,13.626697,176.574293,-1.0,224.816618,-1,-1.0,False,259.230640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,666,1.752887e+09,25,104.868816,81.161743,1,181.025,122.525,0.0,0.0,640.573438,6.572631,92.079744,-1.0,93.916563,-1,-1.0,False,135.183273
1375,666,1.752887e+09,25,104.868816,81.161743,1,149.925,137.075,0.0,0.0,640.573438,7.898041,110.648176,-1.0,115.247464,-1,-1.0,False,162.443790
1376,666,1.752887e+09,25,104.868816,81.161743,1,181.025,137.075,0.0,0.0,640.573438,7.096191,99.414603,-1.0,101.768516,-1,-1.0,False,145.951659
1377,666,1.752887e+09,25,104.868816,81.161743,1,181.025,152.625,0.0,0.0,640.573438,10.918341,152.961277,-1.0,155.851336,-1,-1.0,False,224.564114


In [4]:
a = reco.copy()

In [9]:
a

,event,time,npeak,Xpeak,Ypeak,nsipm,X,Y,Xrms,Yrms,Z,Q,E,Qc,Ec,track_id,Ep,drop,Ec_border
0,29,1.752754e+09,32,-10.893259,25.457517,1,-312.575,-48.025,0.0,0.0,620.592500,11.336324,1007.452496,-1.0,1058.499190,-1,-1.0,False,1479.051964
1,29,1.752754e+09,32,-10.893259,25.457517,1,-297.025,-48.025,0.0,0.0,620.592500,8.926717,793.312129,-1.0,825.987929,-1,-1.0,False,1164.670164
2,29,1.752754e+09,32,-10.893259,25.457517,1,-312.575,-62.575,0.0,0.0,624.352000,7.421574,118.056134,-1.0,124.472592,-1,-1.0,False,173.319494
3,29,1.752754e+09,32,-10.893259,25.457517,1,-297.025,-62.575,0.0,0.0,624.352000,7.558621,120.236167,-1.0,124.710574,-1,-1.0,False,176.520024
4,29,1.752754e+09,32,-10.893259,25.457517,1,-281.475,-78.125,0.0,0.0,624.352000,7.331844,116.628781,-1.0,120.544538,-1,-1.0,False,171.223981
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
925,722,1.752754e+09,24,137.563008,72.314694,1,381.175,184.225,0.0,0.0,253.675375,23.159225,344.142590,-1.0,410.945527,-1,-1.0,False,505.239479
926,722,1.752754e+09,24,137.563008,72.314694,1,396.725,168.675,0.0,0.0,253.675375,7.958416,118.260859,-1.0,139.215811,-1,-1.0,False,173.620053
927,722,1.752754e+09,24,137.563008,72.314694,1,381.175,198.775,0.0,0.0,253.675375,17.003962,252.676299,-1.0,307.183597,-1,-1.0,False,370.956822
928,722,1.752754e+09,24,137.563008,72.314694,1,366.625,184.225,0.0,0.0,257.784625,8.378812,277.776679,-1.0,321.145302,-1,-1.0,False,407.806963


In [56]:
orig_reco = pd.read_hdf(f, 'RECO/Events')

In [17]:
len(reco) / len(reco.event.unique())

1030.142857142857

In [19]:
len(reco.event.unique())

35

In [18]:
len(a) / len(a.event.unique())

743.7096774193549

In [17]:
ev = orig_reco[orig_reco.event == 8]

In [23]:
dropQ, df = make_Q_cut(ev, q_thr)

In [24]:
len(df) + len(dropQ)

1342

In [25]:
len(ev)

1342

In [10]:
reco.Q.min()

5.0

In [163]:
drop_df = reco.groupby(['event', 'npeak'], group_keys=False).apply(dropper)

In [165]:
drop_df.sum()

event       1.180294e+07
time        5.017609e+13
npeak       7.292870e+05
Xpeak       6.712771e+05
Ypeak       1.305047e+06
nsipm       2.862700e+04
X           1.772997e+06
Y           2.618274e+06
Xrms        0.000000e+00
Yrms        0.000000e+00
Z           2.508900e+07
Q           6.369615e+05
E           5.956967e+06
Qc         -2.862700e+04
Ec          3.042123e+01
track_id   -2.862700e+04
Ep         -2.862700e+04
drop        5.338000e+03
dtype: float64

In [158]:
drop_df[drop_df.Q < 7]['drop'] = True

/scratch/888614/ipykernel_3859147/2873677098.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_df[drop_df.Q < 7]['drop'] = True


In [106]:
reco = pd.read_hdf(files[0], 'RECO/Events')

In [107]:
test = reco.groupby(['event', 'npeak'], group_keys=False).apply(dropper)